In [4]:
# ============================================================
# 2. ĐỌC FILE VÀ MATCH GOLD SET VỚI FULL SET
# ============================================================

gold_path = "human_gold_500_final.csv"
full_path = "labeled_data.csv"

gold = pd.read_csv(gold_path)
full = pd.read_csv(full_path)

print("Gold shape:", gold.shape)
print("Full shape:", full.shape)

# Đảm bảo các nhãn đã là số 0/1/2 như bạn đã chuẩn hóa.
gold["final_is_relevant"] = gold["final_is_relevant"].astype(int)
gold["final_manual_label"] = gold["final_manual_label"].astype(int)
full["is_relevant"] = full["is_relevant"].astype(int)
full["manual_label"] = full["manual_label"].astype(int)

# Theo quy ước: irrelevant thì manual_label = 2.
gold.loc[gold["final_is_relevant"] == 0, "final_manual_label"] = 2
full.loc[full["is_relevant"] == 0, "manual_label"] = 2

match_info = match_gold_to_full(gold, full)

print("\nSố gold rows:", len(gold))
print("Số gold rows match được:", int((match_info["matched_full_count"] > 0).sum()))
print("Gold row matched rate:", round(float((match_info["matched_full_count"] > 0).mean()), 4))

# Kiểm tra match theo topic.
matched_topic = []
for _, r in match_info.iterrows():
    if pd.isna(r["matched_full_index"]):
        matched_topic.append(pd.NA)
    else:
        matched_topic.append(full.loc[int(r["matched_full_index"]), "topic"])

match_info["matched_full_topic"] = matched_topic

print("\nGold topic distribution:")
print(gold["topic"].value_counts().sort_index())

print("\nMatched full topic distribution:")
print(match_info["matched_full_topic"].value_counts().sort_index())

print("\nMatch method distribution:")
print(match_info["match_method"].value_counts())

unmatched = match_info[match_info["matched_full_count"] == 0]
print("\nSố gold rows chưa match được:", len(unmatched))
if len(unmatched) > 0:
    display(gold.loc[unmatched["gold_index"], ["id", "topic", "text"]].head(20))


Gold shape: (500, 12)
Full shape: (10718, 14)

Số gold rows: 500
Số gold rows match được: 500
Gold row matched rate: 1.0

Gold topic distribution:
topic
arsenal           100
iphone17series    100
ronaldo           100
s25series         100
vinfast_vf3       100
Name: count, dtype: int64

Matched full topic distribution:
matched_full_topic
arsenal           100
iphone17series    100
ronaldo           100
s25series         100
vinfast_vf3       100
Name: count, dtype: int64

Match method distribution:
match_method
topic+video_id+created_at+text    500
Name: count, dtype: int64

Số gold rows chưa match được: 0


In [5]:
# ============================================================
# 3. TẠO FILE BENCHMARK GỌN
#    Giữ nguyên cột của gold set, chỉ thêm:
#    - ai_is_relevant
#    - ai_manual_label
#    - mismatch_status
# ============================================================

benchmark = gold.copy()

ai_is_relevant = []
ai_manual_label = []
mismatch_status = []

for _, r in match_info.iterrows():
    gold_idx = int(r["gold_index"])

    if pd.isna(r["matched_full_index"]):
        ai_is_relevant.append(pd.NA)
        ai_manual_label.append(pd.NA)
        mismatch_status.append("unmatched")
        continue

    full_idx = int(r["matched_full_index"])

    human_rel = int(gold.loc[gold_idx, "final_is_relevant"])
    human_sent = int(gold.loc[gold_idx, "final_manual_label"])

    ai_rel = int(full.loc[full_idx, "is_relevant"])
    ai_sent = int(full.loc[full_idx, "manual_label"])

    # Ép logic lại cho chắc.
    if ai_rel == 0:
        ai_sent = 2

    ai_is_relevant.append(ai_rel)
    ai_manual_label.append(ai_sent)

    relevance_wrong = human_rel != ai_rel
    sentiment_wrong = (human_rel == 1 and ai_rel == 1 and human_sent != ai_sent)

    if relevance_wrong and sentiment_wrong:
        status = "both_mismatch"
    elif relevance_wrong:
        status = "relevance_mismatch"
    elif sentiment_wrong:
        status = "manual_label_mismatch"
    else:
        status = "match"

    mismatch_status.append(status)

benchmark["ai_is_relevant"] = ai_is_relevant
benchmark["ai_manual_label"] = ai_manual_label
benchmark["mismatch_status"] = mismatch_status

benchmark.to_csv("gold_500_ai_benchmark_matched.csv", index=False, encoding="utf-8-sig")

print("Đã lưu file benchmark gọn: gold_500_ai_benchmark_matched.csv")
print("\nMismatch status:")
print(benchmark["mismatch_status"].value_counts(dropna=False))

display(benchmark.head())


Đã lưu file benchmark gọn: gold_500_ai_benchmark_matched.csv

Mismatch status:
mismatch_status
match                    300
relevance_mismatch       133
manual_label_mismatch     67
Name: count, dtype: int64


,id,video_id,topic,text,clean_text,created_at,source,source_item_id,like_count,reply_count,final_is_relevant,final_manual_label,ai_is_relevant,ai_manual_label,mismatch_status
0,1,0qNL8VZHblA,arsenal,GÁY TO LÊN ANH EM PHÁO THỦ ƠIII,NaN,2026-05-21T11:50:36Z,YouTube,0qNL8VZHblA,0,0,1,0,1,0,match
1,2,0qNL8VZHblA,arsenal,It's not done,NaN,2026-05-20T14:05:33Z,YouTube,0qNL8VZHblA,0,0,0,2,0,2,match
2,3,0qNL8VZHblA,arsenal,Ai vào đây sau khi Arsenal đã vô địch ?,NaN,2026-05-20T05:21:01Z,YouTube,0qNL8VZHblA,0,0,1,2,1,2,match
3,4,0qNL8VZHblA,arsenal,0:21 0:22 MCT là số 1 arsenal tuổi 1:11 1:11,NaN,2026-05-18T11:42:01Z,YouTube,0qNL8VZHblA,0,0,1,1,1,2,manual_label_mismatch
4,5,0qNL8VZHblA,arsenal,"Giờ pháo lại sáng cửa rồi , ngày mai mà thắng ...",NaN,2026-05-09T00:38:20Z,YouTube,0qNL8VZHblA,0,0,1,0,1,0,match


In [1]:
from sklearn.metrics import cohen_kappa_score
import pandas as pd

# ============================================================
# COHEN'S KAPPA GIỮA HUMAN GOLD SET VÀ AI LABEL
# ============================================================

# Nếu trong notebook đã có biến benchmark thì dùng luôn.
# Nếu chưa có, đọc từ file benchmark đã match đúng dòng.
try:
    kappa_df = benchmark.copy()
except NameError:
    kappa_df = pd.read_csv("gold_500_ai_benchmark_matched.csv")

print("========== COHEN'S KAPPA: HUMAN GOLD vs AI ==========")
print("Số dòng benchmark:", len(kappa_df))

# ============================================================
# 1. COHEN KAPPA CHO RELEVANCE
# ============================================================

rel_df = kappa_df.dropna(subset=["final_is_relevant", "ai_is_relevant"]).copy()

kappa_relevance = cohen_kappa_score(
    rel_df["final_is_relevant"],
    rel_df["ai_is_relevant"]
)

print("\n--- Relevance agreement ---")
print("Số mẫu:", len(rel_df))
print("Cohen's Kappa:", round(kappa_relevance, 4))

print("\nBảng đối chiếu relevance:")
print(
    pd.crosstab(
        rel_df["final_is_relevant"],
        rel_df["ai_is_relevant"],
        rownames=["Human final_is_relevant"],
        colnames=["AI is_relevant"]
    )
)

# ============================================================
# 2. COHEN KAPPA CHO SENTIMENT
# Chỉ tính trên các dòng mà cả Human và AI đều xác định là relevant.
# Đây là cách công bằng nhất để đánh giá sentiment riêng.
# ============================================================

sent_df = kappa_df[
    (kappa_df["final_is_relevant"] == 1) &
    (kappa_df["ai_is_relevant"] == 1)
].dropna(subset=["final_manual_label", "ai_manual_label"]).copy()

kappa_sentiment = cohen_kappa_score(
    sent_df["final_manual_label"],
    sent_df["ai_manual_label"]
)

print("\n--- Sentiment agreement, only Human relevant & AI relevant ---")
print("Số mẫu:", len(sent_df))
print("Cohen's Kappa:", round(kappa_sentiment, 4))

print("\nBảng đối chiếu sentiment:")
print(
    pd.crosstab(
        sent_df["final_manual_label"],
        sent_df["ai_manual_label"],
        rownames=["Human final_manual_label"],
        colnames=["AI manual_label"]
    )
)

# ============================================================
# 3. OPTIONAL: SENTIMENT KAPPA TRÊN TOÀN BỘ HUMAN RELEVANT
# Bao gồm cả trường hợp AI dự đoán irrelevant.
# Kết quả này thường thấp hơn và phản ánh lỗi relevance + sentiment gộp chung.
# ============================================================

sent_all_human_relevant = kappa_df[
    kappa_df["final_is_relevant"] == 1
].dropna(subset=["final_manual_label", "ai_manual_label"]).copy()

kappa_sentiment_all_human_relevant = cohen_kappa_score(
    sent_all_human_relevant["final_manual_label"],
    sent_all_human_relevant["ai_manual_label"]
)

print("\n--- Sentiment agreement, all Human relevant comments ---")
print("Số mẫu:", len(sent_all_human_relevant))
print("Cohen's Kappa:", round(kappa_sentiment_all_human_relevant, 4))

print("\nBảng đối chiếu sentiment trên toàn bộ Human relevant:")
print(
    pd.crosstab(
        sent_all_human_relevant["final_manual_label"],
        sent_all_human_relevant["ai_manual_label"],
        rownames=["Human final_manual_label"],
        colnames=["AI manual_label"]
    )
)

========== COHEN'S KAPPA: HUMAN GOLD vs AI ==========
Số dòng benchmark: 500

--- Relevance agreement ---
Số mẫu: 500
Cohen's Kappa: 0.4271

Bảng đối chiếu relevance:
AI is_relevant            0    1
Human final_is_relevant         
0                        96  123
1                        10  271

--- Sentiment agreement, only Human relevant & AI relevant ---
Số mẫu: 271
Cohen's Kappa: 0.6301

Bảng đối chiếu sentiment:
AI manual_label            0   1   2
Human final_manual_label            
0                         71   1   7
1                          8  68  26
2                         13  12  65

--- Sentiment agreement, all Human relevant comments ---
Số mẫu: 281
Cohen's Kappa: 0.6113

Bảng đối chiếu sentiment trên toàn bộ Human relevant:
AI manual_label            0   1   2
Human final_manual_label            
0                         71   1   9
1                          8  68  30
2                         13  12  69


In [6]:
# ============================================================
# 4. BENCHMARK RELEVANCE
# ============================================================

bench_rel = benchmark[benchmark["mismatch_status"] != "unmatched"].copy()

y_true_rel = bench_rel["final_is_relevant"].astype(int)
y_pred_rel = bench_rel["ai_is_relevant"].astype(int)

print("========== RELEVANCE BENCHMARK ==========")
print("Samples:", len(bench_rel))
print("Accuracy:", accuracy_score(y_true_rel, y_pred_rel))
print("Macro-F1:", f1_score(y_true_rel, y_pred_rel, average="macro"))

print("\nClassification report:")
print(classification_report(
    y_true_rel,
    y_pred_rel,
    labels=[0, 1],
    target_names=["human_irrelevant", "human_relevant"],
    digits=4
))

cm_rel = pd.DataFrame(
    confusion_matrix(y_true_rel, y_pred_rel, labels=[0, 1]),
    index=["Human irrelevant", "Human relevant"],
    columns=["AI irrelevant", "AI relevant"]
)

print("\nConfusion matrix:")
display(cm_rel)


========== RELEVANCE BENCHMARK ==========
Samples: 500
Accuracy: 0.734
Macro-F1: 0.6968660968660969

Classification report:
                  precision    recall  f1-score   support

human_irrelevant     0.9057    0.4384    0.5908       219
  human_relevant     0.6878    0.9644    0.8030       281

        accuracy                         0.7340       500
       macro avg     0.7967    0.7014    0.6969       500
    weighted avg     0.7832    0.7340    0.7100       500


Confusion matrix:


,AI irrelevant,AI relevant
Human irrelevant,96,123
Human relevant,10,271


In [7]:
# ============================================================
# 5. XUẤT FILE RELEVANCE MISMATCH GỌN
# ============================================================

relevance_mismatches = benchmark[
    (benchmark["mismatch_status"] != "unmatched")
    & (benchmark["final_is_relevant"].astype(int) != benchmark["ai_is_relevant"].astype(int))
].copy()

relevance_mismatches.to_csv(
    "relevance_mismatches_matched.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu: relevance_mismatches_matched.csv")
print("Số relevance mismatches:", len(relevance_mismatches))
print("\nTheo topic:")
print(relevance_mismatches["topic"].value_counts().sort_index())

display(relevance_mismatches.head(20))


Đã lưu: relevance_mismatches_matched.csv
Số relevance mismatches: 133

Theo topic:
topic
arsenal           17
iphone17series    24
ronaldo           22
s25series         32
vinfast_vf3       38
Name: count, dtype: int64


,id,video_id,topic,text,clean_text,created_at,source,source_item_id,like_count,reply_count,final_is_relevant,final_manual_label,ai_is_relevant,ai_manual_label,mismatch_status
8,9,0qNL8VZHblA,arsenal,5:17 đảo đảo bóng nó cay cú nó tán khứa Cherki...,NaN,2026-04-25T17:21:19Z,YouTube,0qNL8VZHblA,0,0,0,2,1,0,relevance_mismatch
16,17,0qNL8VZHblA,arsenal,Cúp nè MC tới lấy đi,NaN,2026-04-22T11:55:49Z,YouTube,0qNL8VZHblA,0,0,0,2,1,1,relevance_mismatch
33,34,0qNL8VZHblA,arsenal,BLV fan Arsenal à ? Erling Halan ghi bàn hay v...,NaN,2026-04-21T11:10:02Z,YouTube,0qNL8VZHblA,0,0,1,2,0,2,relevance_mismatch
39,40,0qNL8VZHblA,arsenal,"Cặp CB của Mci uy tín quá, nếu ko có sai lầm c...",NaN,2026-04-21T05:01:21Z,YouTube,0qNL8VZHblA,0,0,1,1,0,2,relevance_mismatch
44,45,0qNL8VZHblA,arsenal,PHAO THU CHI ĐC LUC ĐÂU THÔI MAN CITY THI ĐI ...,NaN,2026-04-21T02:57:56Z,YouTube,0qNL8VZHblA,0,0,1,1,0,2,relevance_mismatch
45,46,0qNL8VZHblA,arsenal,2 ông BLV fan phấn à mà đến luýc bl phần của p...,NaN,2026-04-21T02:11:37Z,YouTube,0qNL8VZHblA,0,0,1,2,0,2,relevance_mismatch
47,48,0qNL8VZHblA,arsenal,Asean hay nhưng tiếc la ko được may,NaN,2026-04-21T00:52:03Z,YouTube,0qNL8VZHblA,0,0,1,2,0,2,relevance_mismatch
59,60,0qNL8VZHblA,arsenal,Phải loại thằng thủ môn Doraruma lâu rồi mới đ...,NaN,2026-04-20T16:51:40Z,YouTube,0qNL8VZHblA,0,0,0,2,1,1,relevance_mismatch
67,68,0qNL8VZHblA,arsenal,Chịu,NaN,2026-04-20T15:17:57Z,YouTube,0qNL8VZHblA,0,0,0,2,1,1,relevance_mismatch
78,79,0qNL8VZHblA,arsenal,đã kết thúc mùa đâu bạn 😅,NaN,2026-04-22T07:58:24Z,YouTube,0qNL8VZHblA,0,0,0,2,1,2,relevance_mismatch


In [8]:
# ============================================================
# 6. BENCHMARK SENTIMENT / MANUAL_LABEL
#    Chỉ tính trên những dòng human relevant và AI cũng relevant.
# ============================================================

sent_bench = benchmark[
    (benchmark["mismatch_status"] != "unmatched")
    & (benchmark["final_is_relevant"].astype(int) == 1)
    & (benchmark["ai_is_relevant"].astype(int) == 1)
].copy()

y_true_sent = sent_bench["final_manual_label"].astype(int)
y_pred_sent = sent_bench["ai_manual_label"].astype(int)

print("========== SENTIMENT / MANUAL_LABEL BENCHMARK ==========")
print("Samples:", len(sent_bench))
print("Accuracy:", accuracy_score(y_true_sent, y_pred_sent))
print("Macro-F1:", f1_score(y_true_sent, y_pred_sent, average="macro"))

print("\nClassification report:")
print(classification_report(
    y_true_sent,
    y_pred_sent,
    labels=[0, 1, 2],
    target_names=["positive", "negative", "neutral"],
    digits=4
))

cm_sent = pd.DataFrame(
    confusion_matrix(y_true_sent, y_pred_sent, labels=[0, 1, 2]),
    index=["Human positive", "Human negative", "Human neutral"],
    columns=["AI positive", "AI negative", "AI neutral"]
)

print("\nConfusion matrix:")
display(cm_sent)


========== SENTIMENT / MANUAL_LABEL BENCHMARK ==========
Samples: 271
Accuracy: 0.7527675276752768
Macro-F1: 0.7550227057781259

Classification report:
              precision    recall  f1-score   support

    positive     0.7717    0.8987    0.8304        79
    negative     0.8395    0.6667    0.7432       102
     neutral     0.6633    0.7222    0.6915        90

    accuracy                         0.7528       271
   macro avg     0.7582    0.7625    0.7550       271
weighted avg     0.7612    0.7528    0.7514       271


Confusion matrix:


,AI positive,AI negative,AI neutral
Human positive,71,1,7
Human negative,8,68,26
Human neutral,13,12,65


In [9]:
# ============================================================
# 7. XUẤT FILE MANUAL_LABEL MISMATCH GỌN
#    Chỉ xét trên những dòng human relevant và AI relevant.
# ============================================================

manual_label_mismatches = sent_bench[
    sent_bench["final_manual_label"].astype(int) != sent_bench["ai_manual_label"].astype(int)
].copy()

manual_label_mismatches.to_csv(
    "manual_label_mismatches_matched.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu: manual_label_mismatches_matched.csv")
print("Số manual_label mismatches:", len(manual_label_mismatches))
print("\nTheo topic:")
print(manual_label_mismatches["topic"].value_counts().sort_index())

display(manual_label_mismatches.head(20))


Đã lưu: manual_label_mismatches_matched.csv
Số manual_label mismatches: 67

Theo topic:
topic
arsenal           17
iphone17series    20
ronaldo           14
s25series          6
vinfast_vf3       10
Name: count, dtype: int64


,id,video_id,topic,text,clean_text,created_at,source,source_item_id,like_count,reply_count,final_is_relevant,final_manual_label,ai_is_relevant,ai_manual_label,mismatch_status
3,4,0qNL8VZHblA,arsenal,0:21 0:22 MCT là số 1 arsenal tuổi 1:11 1:11,NaN,2026-05-18T11:42:01Z,YouTube,0qNL8VZHblA,0,0,1,1,1,2,manual_label_mismatch
9,10,0qNL8VZHblA,arsenal,Lách qua khe cửa hẹp pháo vẫn quyết định được ...,NaN,2026-04-25T10:46:46Z,YouTube,0qNL8VZHblA,0,0,1,1,1,0,manual_label_mismatch
17,18,0qNL8VZHblA,arsenal,Trận này Ars đá hay hơn,NaN,2026-04-22T02:02:44Z,YouTube,0qNL8VZHblA,0,0,1,0,1,2,manual_label_mismatch
18,19,0qNL8VZHblA,arsenal,Ars thời Giáo Sư tôi yêu \nCòn bjo là màu xanh...,NaN,2026-04-22T01:53:47Z,YouTube,0qNL8VZHblA,0,0,1,2,1,0,manual_label_mismatch
42,43,0qNL8VZHblA,arsenal,Chuông kì bu thì đá mấy vẫn vậy th à = )))))))),NaN,2026-04-21T04:30:28Z,YouTube,0qNL8VZHblA,0,0,1,1,1,2,manual_label_mismatch
55,56,0qNL8VZHblA,arsenal,Kẹt pháo r kẹt pháo r😂,NaN,2026-04-20T17:36:42Z,YouTube,0qNL8VZHblA,0,0,1,1,1,0,manual_label_mismatch
57,58,0qNL8VZHblA,arsenal,"etita là người rất giỏi, giỏi trong những trận...",NaN,2026-04-20T17:24:07Z,YouTube,0qNL8VZHblA,0,0,1,1,1,2,manual_label_mismatch
71,72,0qNL8VZHblA,arsenal,Anh Phú Ngao đó à 😂,NaN,2026-04-20T15:03:15Z,YouTube,0qNL8VZHblA,0,0,1,1,1,2,manual_label_mismatch
72,73,0qNL8VZHblA,arsenal,Không thấy bọn fan phú ngao vào comment ta?,NaN,2026-04-20T15:02:15Z,YouTube,0qNL8VZHblA,0,0,1,1,1,2,manual_label_mismatch
73,74,0qNL8VZHblA,arsenal,Phú Ngao thứ 2 thì không ai dám giành. Cú ăn 4...,NaN,2026-04-20T15:00:19Z,YouTube,0qNL8VZHblA,0,0,1,1,1,2,manual_label_mismatch
